In [81]:
!pip -q install transformers datasets accelerate sentencepiece

In [83]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [84]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [85]:
def prepare_text(df):

    df = df.copy()

    df["text"] = (
        "Question:\n"
        + df["prompt"].fillna("")
        + "\n\nChoices:\n"
        + "A. " + df["A"].fillna("") + "\n"
        + "B. " + df["B"].fillna("") + "\n"
        + "C. " + df["C"].fillna("") + "\n"
        + "D. " + df["D"].fillna("") + "\n"
        + "E. " + df["E"].fillna("") + "\n\n"
        + "Select the correct answer."
    )

    return df


train_df = prepare_text(train_df)
test_df = prepare_text(test_df)

train_df["label"] = train_df["answer"].map({
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
})

In [86]:
MODEL_NAME = "google/electra-base-discriminator"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

print("Loaded Successfully!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Loaded Successfully!


In [87]:
train_data, valid_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

In [90]:
train_dataset = Dataset.from_pandas(
    train_data[["text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_data[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [91]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=384
    )

In [92]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [93]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask"
    ]
)

In [94]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = (predictions == labels).mean()

    return {
        "accuracy": accuracy
    }

In [95]:
training_args = TrainingArguments(

    output_dir="./electra_mcq",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    warmup_steps=100,

    weight_decay=0.01,

    load_best_model_at_end=True,

    report_to="none"
)

In [96]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [97]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.590096,0.630000
2,No log,0.440169,0.990000
3,No log,0.028634,1.000000
4,No log,0.012057,1.000000
5,1.082531,0.009837,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=565, training_loss=0.9598795441399633, metrics={'train_runtime': 442.0338, 'train_samples_per_second': 20.36, 'train_steps_per_second': 1.278, 'total_flos': 1776047461632000.0, 'train_loss': 0.9598795441399633, 'epoch': 5.0})

In [98]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.009836784563958645,
 'eval_accuracy': 1.0,
 'eval_runtime': 3.3744,
 'eval_samples_per_second': 59.27,
 'eval_steps_per_second': 3.853,
 'epoch': 5.0}

In [100]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

In [101]:
import numpy as np
import torch

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

labels = np.array(["A", "B", "C", "D", "E"])

top3 = np.argsort(
    probs,
    axis=1
)[:, -3:][:, ::-1]

predictions = [
    " ".join(labels[idx])
    for idx in top3
]

In [102]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,A C B
1,2,B D E
2,3,B A D
3,4,E C B
4,5,C D A


In [104]:
import os
import shutil
import pandas as pd

assert os.path.exists("submission.csv"), "submission.csv not found!"

submission = pd.read_csv("submission.csv")
print(submission.head())

os.makedirs("/kaggle/working/final_submission", exist_ok=True)
shutil.copy("submission.csv", "/kaggle/working/final_submission/submission.csv")

print("Saved successfully!")

   id prediction
0   1      A C B
1   2      B D E
2   3      B A D
3   4      E C B
4   5      C D A
Saved successfully!


In [105]:
import os

print(os.listdir("/kaggle/working"))
print(os.listdir("/kaggle/working/final_submission"))

['electra_mcq', '.virtual_documents', 'modernbert_mcq', 'final_submission', 'submission.csv']
['submission.csv']
